In [ ]:
!pip install ipytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 39.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ipytest]


In [ ]:
!pip install ruff pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 17.6 MB/s  0:00:00


In [ ]:
def prosjecna_cijena(cijene: list[float]) -> float:
    """
    Računa prosječnu cijenu iz liste cijena.
    Ako je lista prazna, vraća 0.0.
    """
    if not cijene:
        return 0.0
    return sum(cijene) / len(cijene)

# Brzi test da vidimo da radi
print(prosjecna_cijena([10, 20, 30]))

20.0


In [ ]:
%%writefile test_zadatak.py
from __main__ import prosjecna_cijena

def test_prosjecna_cijena_normalne_vrijednosti():
    assert prosjecna_cijena([10.0, 20.0, 30.0]) == 20.0

def test_prosjecna_cijena_jedan_element():
    assert prosjecna_cijena([15.5]) == 15.5

def test_prosjecna_cijena_prazna_lista():
    assert prosjecna_cijena([]) == 0.0

Writing test_zadatak.py


In [ ]:
!pytest test_zadatak.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
_______________________ ERROR collecting test_zadatak.py _______________________
ImportError while importing test module '/content/test_zadatak.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
test_zadatak.py:1: in <module>
    from __main__ import prosjecna_cijena
E   ImportError: cannot import name 'prosjecna_cijena' from '__

In [ ]:
import ipytest
ipytest.autoconfig()

def prosjecna_cijena(cijene: list[float]) -> float:
    if not cijene:
        return 0.0
    return sum(cijene) / len(cijene)

# Testovi idu ispod, u istoj ćeliji
def test_prosjecna_cijena_normalne_vrijednosti():
    assert prosjecna_cijena([10.0, 20.0, 30.0]) == 20.0

def test_prosjecna_cijena_jedan_element():
    assert prosjecna_cijena([15.5]) == 15.5

def test_prosjecna_cijena_prazna_lista():
    assert prosjecna_cijena([]) == 0.0

# Pokretanje testova
ipytest.run("-v")

======================================= test session starts ========================================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 3 items

t_8818329eb558470a860ffad0b960ce60.py ...                                                    [100%]

======================================== 3 passed in 0.02s =========================================


<ExitCode.OK: 0>

In [1]:
from typing import Protocol, List, Dict, Any

# --- PROTOKOLI (Dependency Inversion) ---
class RezervacijaRepository(Protocol):
    def sacuvaj(self, rezervacija: Dict[str, Any]) -> None: ...
    def vrati_sve(self) -> List[Dict[str, Any]]: ...

class NotifikacioniServis(Protocol):
    def posalji_potvrdu(self, email: str, detalji: str) -> None: ...

# --- POSLOVNA LOGIKA (Single Responsibility) ---
class ObradaRezervacija:
    def __init__(self, repo: RezervacijaRepository, notifikator: NotifikacioniServis):
        self.repo = repo
        self.notifikator = notifikator

    def kreiraj_rezervaciju(self, gost: str, email: str, soba: str, nocenja: int) -> Dict[str, Any]:
        nova_rezervacija = {"gost": gost, "email": email, "soba": soba, "nocenja": nocenja}
        self.repo.sacuvaj(nova_rezervacija)
        self.notifikator.posalji_potvrdu(email, f"Rezervisali ste sobu {soba} na {nocenja} noćenja.")
        return nova_rezervacija

    def generisi_izvjestaj(self) -> str:
        sve_rezervacije = self.repo.vrati_sve()
        ukupno_nocenja = sum(r["nocenja"] for r in sve_rezervacije)
        return f"Ukupan broj rezervacija: {len(sve_rezervacije)}. Ukupno noćenja: {ukupno_nocenja}."

In [2]:
class InMemoryRezervacijaRepository:
    def __init__(self):
        self.rezervacije = []
    def sacuvaj(self, rezervacija: Dict[str, Any]) -> None:
        self.rezervacije.append(rezervacija)
    def vrati_sve(self) -> List[Dict[str, Any]]:
        return self.rezervacije

class FakeEmailServis:
    def __init__(self):
        self.poslate_poruke = []
    def posalji_potvrdu(self, email: str, detalji: str) -> None:
        self.poslate_poruke.append({"email": email, "detalji": detalji})

In [3]:
def pokreni_testove():
    print("Pokrećem testove...\n")

    # Test 1
    repo1 = InMemoryRezervacijaRepository()
    notif1 = FakeEmailServis()
    servis1 = ObradaRezervacija(repo1, notif1)
    servis1.kreiraj_rezervaciju("Ivan", "ivan@test.com", "101", 3)
    assert len(repo1.rezervacije) == 1
    assert repo1.rezervacije[0]["gost"] == "Ivan"
    assert len(notif1.poslate_poruke) == 1
    assert notif1.poslate_poruke[0]["email"] == "ivan@test.com"
    print("Test 1 (Kreiranje rezervacije): PROŠAO")

    # Test 2
    repo2 = InMemoryRezervacijaRepository()
    notif2 = FakeEmailServis()
    servis2 = ObradaRezervacija(repo2, notif2)
    servis2.kreiraj_rezervaciju("Marko", "marko@test.com", "102", 2)
    servis2.kreiraj_rezervaciju("Jovana", "jovana@test.com", "103", 4)
    izvjestaj = servis2.generisi_izvjestaj()
    assert "Ukupan broj rezervacija: 2" in izvjestaj
    assert "Ukupno noćenja: 6" in izvjestaj
    print("Test 2 (Generisanje izveštaja): PROŠAO")

    # Test 3
    repo3 = InMemoryRezervacijaRepository()
    notif3 = FakeEmailServis()
    servis3 = ObradaRezervacija(repo3, notif3)
    izvjestaj_prazan = servis3.generisi_izvjestaj()
    assert "Ukupan broj rezervacija: 0" in izvjestaj_prazan
    print("Test 3 (Prazan izveštaj): PROŠAO")

    print("\nSVI TESTOVI SU USPEŠNO PROŠLI!")

pokreni_testove()

Pokrećem testove...

Test 1 (Kreiranje rezervacije): PROŠAO
Test 2 (Generisanje izveštaja): PROŠAO
Test 3 (Prazan izveštaj): PROŠAO

SVI TESTOVI SU USPEŠNO PROŠLI!
